# Independent assembly comparison

This notebook compares four already generated H5AD assemblies (mean, random, within_cluster, and outside_cluster). It never reconstructs or writes back to a native H5AD. The current explicit broad types are T, Mono_Macro, and Fibroblast.

For each type, exact real IDs and exact gene names define the common scope. Every method is clustered independently. Each method's own finite spatial coordinates are aligned by ID and checked for exact equality with the baseline before plotting. No resolution or method is selected as a winner.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from revise.analysis.assembly_comparison import (
    INPUT_ASSUMPTIONS,
    DEFAULT_CELL_TYPES,
    assert_input_hashes_unchanged,
    compare_assembly_methods,
    input_summary_frame,
    load_assembly_inputs,
    plot_spatial_comparison,
)

PARAMETERS = {
    'method_paths': {
        'mean': 'output/assembly/mean.h5ad',
        'random': 'output/assembly/random.h5ad',
        'within_cluster': 'output/assembly/within_cluster.h5ad',
        'outside_cluster': 'output/assembly/outside_cluster.h5ad',
    },
    'baseline_path': 'output/assembly/original_spatial.h5ad',
    'cell_types': list(DEFAULT_CELL_TYPES),
    'broad_column': 'Level1',
    'baseline_subtype_column': 'SVC_cluster',
    'spatial_key': 'spatial',
    'coordinate_unit': 'from formal sample.yaml and baseline provenance',
    'microns_per_coordinate': 0.2125,
    'seed': 42,
    'resolutions': [0.6, 0.7, 0.8],
    'plot_resolution': 0.7,
    'output_dir': 'output/assembly/comparison',
}
config_path = os.environ.get('REVISE_ASSEMBLY_COMPARISON_CONFIG')
if config_path:
    PARAMETERS.update(json.loads(Path(config_path).read_text(encoding='utf-8')))
PARAMETERS['cell_types'] = list(PARAMETERS['cell_types'])
output_dir = Path(PARAMETERS.get('output_dir', 'output/assembly/comparison')).expanduser()
output_dir.mkdir(parents=True, exist_ok=True)
PARAMETERS

## 1. Declare the comparison boundary

Generated matrices are declared finite, nonnegative, unlogged linear expression. Fractional mixtures remain valid. The helper validates this boundary before making working copies. Broad labels use exact explicit values after only slash-to-underscore normalization; missing values remain missing.

In [ ]:
assumption_table = pd.Series(INPUT_ASSUMPTIONS, name='declared behavior').to_frame()
display(assumption_table)

## 2. Read native H5ADs and fingerprint them

Hashes are captured before analysis and checked again at the end. Shapes below describe native files before type selection or intersection.

In [ ]:
inputs = load_assembly_inputs(
    PARAMETERS['method_paths'],
    PARAMETERS['baseline_path'],
)
input_summary = input_summary_frame(inputs)
display(input_summary)

## 3. Build equal scopes, then cluster each method independently

For a broad type, every method and the baseline are intersected on real IDs. Expression methods are also intersected on exact genes. Each method then receives its own normalization, log1p, PCA, neighbor graph, and Leiden run. Baseline labels and coordinates are not inputs to expression clustering.

In [ ]:
comparison = compare_assembly_methods(
    inputs.methods,
    inputs.baseline,
    cell_types=PARAMETERS['cell_types'],
    broad_column=PARAMETERS['broad_column'],
    baseline_subtype_column=PARAMETERS['baseline_subtype_column'],
    spatial_key=PARAMETERS['spatial_key'],
    resolutions=PARAMETERS['resolutions'],
    seed=PARAMETERS['seed'],
)
display(comparison.coverage)

In [ ]:
scope_summary = pd.DataFrame([
    {
        'broad_type': name,
        'status': result.status,
        'issues': '; '.join(result.issues) or None,
        'shared_id_count': len(result.shared_ids),
        'shared_gene_count': len(result.shared_genes),
        'shared_id_preview': tuple(result.shared_ids[:5]),
        'shared_gene_preview': tuple(result.shared_genes[:10]),
    }
    for name, result in comparison.by_type.items()
])
display(scope_summary)

## 4. Read metrics and label correspondences

ARI and NMI use only exact shared IDs whose baseline subtype label is present. Matched and missing label counts remain beside every score. Contingency tables retain label-to-cluster structure; cluster numbers are method-local. All requested resolutions stay visible.

In [ ]:
display(comparison.metrics)
for broad_type, result in comparison.by_type.items():
    if result.status != 'ok':
        print(f'{broad_type}: unavailable — {"; ".join(result.issues)}')
        continue
    for method in inputs.methods:
        for resolution in PARAMETERS['resolutions']:
            print(f'{broad_type} | {method} | resolution={resolution}')
            display(result.contingencies[(method, float(resolution))])

## 5. Inspect spatial organization at one declared resolution

Each plot uses the method's own coordinates only after the ID-aligned shape, finite-value, and exact-equality check against the baseline. The left panel shows the baseline subtype field and the right panel shows the method-local expression Leiden labels.

In [ ]:
plot_resolution = float(PARAMETERS['plot_resolution'])
for broad_type, result in comparison.by_type.items():
    if result.status != 'ok':
        continue
    for method in inputs.methods:
        figure, _ = plot_spatial_comparison(
            result,
            inputs.baseline,
            method=method,
            resolution=plot_resolution,
            spatial_key=PARAMETERS['spatial_key'],
        )
        figure.savefig(
            output_dir / f'{broad_type}-{method}-r{plot_resolution:g}.png',
            dpi=150,
            bbox_inches='tight',
        )
        display(figure)
        plt.close(figure)

## 6. Save the thin comparison report

The report keeps effective parameters, native input hashes, coverage, metrics, every requested contingency table, and the rendered plots. It does not choose a winner.

In [ ]:
effective_parameters = dict(PARAMETERS)
effective_parameters['output_dir'] = str(output_dir)
(output_dir / 'effective_parameters.json').write_text(
    json.dumps(effective_parameters, indent=2, sort_keys=True),
    encoding='utf-8',
)
input_summary.to_csv(output_dir / 'input_summary.csv', index=False)
baseline_provenance = inputs.baseline.uns.get('assembly_baseline', {})
(output_dir / 'baseline_provenance.json').write_text(
    json.dumps(baseline_provenance, indent=2, sort_keys=True, default=str),
    encoding='utf-8',
)
scope_summary.to_csv(output_dir / 'scope_summary.csv', index=False)
comparison.coverage.to_csv(output_dir / 'coverage.csv', index=False)
comparison.metrics.to_csv(output_dir / 'metrics.csv', index=False)
contingency_dir = output_dir / 'contingencies'
contingency_dir.mkdir(parents=True, exist_ok=True)
for broad_type, result in comparison.by_type.items():
    for (method, resolution), table in result.contingencies.items():
        table.to_csv(
            contingency_dir / f'{broad_type}-{method}-r{float(resolution):g}.csv'
        )
display(pd.Series({'report_dir': str(output_dir), 'contingency_tables': sum(len(r.contingencies) for r in comparison.by_type.values())}))

## 7. Confirm native inputs stayed unchanged

Any native H5AD change is an error rather than an analysis output.

In [ ]:
assert_input_hashes_unchanged(inputs)
print('native input hashes unchanged')